# Orbit graph analysis — fast pipeline

One notebook for the whole per-orbit analysis, for **any prime `d` and any `n_max`**: set the knobs in the next cell, run top to bottom, and get the paper's per-orbit table (CSV + LaTeX longtable) plus validation and the estimator figure.

Backed by:
1. **`orbits_mt.c`** (run separately, once, to generate the data) — writes deduplicated edge lists: each unordered pair on one line (from its smaller endpoint), one self-loop line per fixed graph. E.g. `./orbits_mt 7 3 --upto --outdir-fmt "orbits_d3_n%d_separated" --loghash 23`.
2. **`fast_analysis.py`** — orbit-graph (OG) observables on a scipy CSR adjacency: $|V|$, density $\mathcal{D}$, $\langle d_{OG}\rangle$ (exact when the all-sources BFS pass fits `BUDGET`, else source-sampled with a standard error), $d_{OG}^{\max}$, $\deg(OG)_{\max}$, $N_\mathcal{L}$, greedy $\chi_{OG}$. All BFS goes through one chunked, threaded reducer that never materialises a distance matrix.
3. **`orbit_scan.c`** (compiled automatically via `scan_tool.py`) — one streaming C pass per level for the member-side observables: the representative (minimal under #edges, total weight, code), $|e|$, $\chi_i$, $\deg(g)_{\min}$ (minimal member max **weighted** degree, App. B of the paper), and the Schmidt bounds.
4. **`make_table.py`** — joins 2 and 3, applies the paper ordering (n ascending, then representative #edges, total weight, code) and writes the longtable. The expensive OG statistics are cached in `PREFIX_stats_cache.csv`, so an interrupted run resumes where it stopped.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from fast_analysis import load_orbit, aspl_sampled
from make_table import gather, number_orbits, latex_table

# ---- knobs: everything below is driven by these --------------------------
d       = 5
n_max   = 6                       # needs orbit dirs for every n = 3..n_max
ROOT    = "data"                   # directory containing the orbit dirs
DIR_FMT = "orbits_d{d}_n{n}_separated"   # takes {d} and {n}
WORKERS = os.cpu_count()           # orbits processed in parallel (processes)
BUDGET  = 20.0                     # single-core seconds allowed for an exact all-sources pass
SOURCES = 1000                     # BFS sources when an orbit falls back to sampling
PREFIX  = f"table_d{d}_n{n_max}"   # outputs: PREFIX.csv, PREFIX.tex, PREFIX_stats_cache.csv

orbit_dir = lambda n: os.path.join(ROOT, DIR_FMT.format(d=d, n=n))

## Everything per orbit and the LaTeX table
`gather` runs the C scanner over levels 3..`n_max` (members, representative, $\chi_i$, $\deg(g)_{\min}$, Schmidt bounds) and computes the orbit-graph statistics per orbit (exact within `BUDGET`, else sampled — `aspl_se == 0.0` marks exact rows, `diameter_exactly_known` marks exact diameters). Uncached orbits run `WORKERS` at a time in separate processes, biggest first, and each finished orbit is cached immediately — so progress lines arrive in completion order and re-running is cheap.

In [ ]:
df = gather(d, n_max, DIR_FMT, ROOT, BUDGET, WORKERS, SOURCES,
            PREFIX + "_stats_cache.csv")
df = number_orbits(df)
df.to_csv(PREFIX + ".csv", index=False)
latex_table(df, d, PREFIX + ".tex")

exact = int((df.es_lo == df.es_up).sum())
print(f"{len(df)} orbits, {df.members.sum()} graphs; "
      f"E_S exact for {exact}, interval for {len(df) - exact}")
print(f"wrote {PREFIX}.csv and {PREFIX}.tex")
df.head()

In [ ]:
# Legacy per-metric text files (layout the old notebooks consumed).
# number_of_loops is N_L = distinct fixed graphs (loop_nodes), as in the paper.
out = f"og_data_fast_d{d}_n{n_max}"
os.makedirs(out, exist_ok=True)
np.savetxt(f"{out}/aspl.txt", df["aspl"].values)
np.savetxt(f"{out}/diameter.txt", df["diameter"].values)
np.savetxt(f"{out}/density.txt", df["density"].values)
np.savetxt(f"{out}/max_degree.txt", df["max_degree"].values)
np.savetxt(f"{out}/number_of_loops.txt", df["loop_nodes"].values)
df.to_csv(f"{out}/og_stats.csv", index=False)
print("wrote", out)